In [1]:
#%%time
import os
import re
import gzip
import math
import random
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad

from ast import literal_eval
import warnings
# 忽略 FutureWarning 类型警告
warnings.simplefilter(action='ignore', category=FutureWarning)
# 忽略特定类型的警告：忽略 scanpy包中含有 ignore 的 UserWarning 类型警告
warnings.filterwarnings("ignore", category=UserWarning, module="scanpy")
# 禁用 pandas 包中的 SettingWithCopyWarning 类型警告  
pd.options.mode.chained_assignment = None  # 或 'raise' 表示引发异常

import inferECC
from inferECC import *

species value: hg38
species == 'mm10':  False


In [2]:
import pandas as pd
from pyliftover import LiftOver

# 初始化 liftover 对象
lo = LiftOver('hg19', 'hg38')  # 第一次运行会自动下载 chain 文件

# 示例 DataFrame（你可以换成自己的实际数据）
df = pd.DataFrame({
    'chr_100k': ['chr1:0_100000', 'chr2:100000_200000', 'chrY:59000000_59100000']
})

# 定义转换函数
def convert_interval(interval):
    try:
        chrom, pos_range = interval.split(':')
        start_str, end_str = pos_range.split('_')
        start, end = int(start_str), int(end_str)

        # 处理起点为0的特殊情况，改为1尝试转换
        start_to_convert = 1 if start == 0 else start

        res_start = lo.convert_coordinate(chrom, start_to_convert)
        res_end = lo.convert_coordinate(chrom, end - 1)  # 闭区间

        # 处理各种转换失败的情况：
        if not res_start and not res_end:
            # 两者都失败，返回原区间
            return interval
        elif not res_start and res_end:
            # start失败，end成功：start = end转换后 - 100000
            new_chrom = res_end[0][0]
            new_end = res_end[0][1] + 1
            new_start = max(new_end - 100000, 0)
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"
        elif res_start and not res_end:
            # end失败，start成功：end = start转换后 + 100000
            new_chrom = res_start[0][0]
            new_start = res_start[0][1] - 1 if start == 0 else res_start[0][1]
            new_start = max(new_start, 0)
            new_end = new_start + 100000
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"
        else:
            # 两者都成功
            new_chrom = res_start[0][0]
            new_start = res_start[0][1] - 1 if start == 0 else res_start[0][1]
            new_start = max(new_start, 0)
            new_end = res_end[0][1] + 1
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"
    except Exception as e:
        print(f"Error processing interval {interval}: {e}")
        return None

# 应用转换函数到每一行
df['chr_100k_hg38'] = df['chr_100k'].apply(convert_interval)

# 输出结果
print(df)


                 chr_100k           chr_100k_hg38
0           chr1:0_100000           chr1:0_100000
1      chr2:100000_200000      chr2:100000_200000
2  chrY:59000000_59100000  chrY:56853853_56953852


In [3]:
import pandas as pd

# 示例 df
df = pd.DataFrame({
    'colx': ['chr1:0_100000', 'chr2:100000_200000', 'chr17:53000000_53010000', 'chr8:127000000_127010000']
})

# POSITIVE 区间列表
POS_LIST = [
    'chr2:54448152_58624587',
    'chr2:118483904_118488904',
    'chr20:7293703_8236529',
    'chr2:51804053_54073023',
    'chr17:52106827_54486261',
    'chr21:5216245_5255990',
    'chr8:126424717_127997899'
]

# 将 POS_LIST 转换为标准区间列表 (chr, start, end)
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return (chrom, int(start_str), int(end_str))

positive_intervals = [parse_region(pos) for pos in POS_LIST]

# 判断一个区域是否与任意 positive 区域重叠
def has_overlap(region_str):
    chrom, coords = region_str.split(':')
    start, end = map(int, coords.split('_'))

    for pos_chr, pos_start, pos_end in positive_intervals:
        if chrom == pos_chr:
            # 判断是否有重叠（交集）：不是完全左或右就算交集
            if not (end < pos_start or start > pos_end):
                return True
    return False

# 应用判断函数，添加新列
df['is_positive'] = df['colx'].apply(has_overlap)

# 查看结果
print(df)


                       colx  is_positive
0             chr1:0_100000        False
1        chr2:100000_200000        False
2   chr17:53000000_53010000         True
3  chr8:127000000_127010000         True


In [4]:
# 1.下载已报道各细胞系CNV、ECDNA区域数据
# 2.识别两者的fragments，统计数量，剩下的就是BG
# 3.三者分别统计密度分布
# 4.三者统计中位数分布，差异检验

In [5]:
# COLO320DM
# PC3
# 

In [6]:
import pandas as pd

# 读取 Excel 文件中的 "COLO" sheet
#df = pd.read_excel('E:/05.project/04.ecDNA/01.data/scCircle-seq/FIG2.xlsx', sheet_name='Colo320DM', index_col=None)
df = pd.read_excel('E:/05.project/04.ecDNA/01.data/scCircle-seq/FIG2.xlsx', sheet_name='PC3', index_col=None)
df

,CHR,POS1,POS2,FREQUENCY,UNIFORMITY,TYPE
0,chr1,5262241,5290853,0.000000,0.000000,Low frequency unstable circle(LU)
1,chr1,6682911,6714302,0.000000,0.000000,Low frequency unstable circle(LU)
2,chr1,8074266,8101772,0.000000,0.000000,Low frequency unstable circle(LU)
3,chr1,12191185,12227871,0.000000,0.000000,Low frequency unstable circle(LU)
4,chr1,15536234,15563379,0.000000,0.000000,Low frequency unstable circle(LU)
...,...,...,...,...,...,...
13066,chr8,133659488,133874959,0.384935,0.327195,High frequency stable circle(HS)
13067,chr8,130566097,130910958,0.337267,0.303541,High frequency stable circle(HS)
13068,chr8,121845273,121996460,0.375357,0.356589,High frequency stable circle(HS)
13069,chr8,128547070,128647504,0.372092,0.372092,High frequency stable circle(HS)


In [7]:
df["region"] = df["CHR"]+":"+(df["POS1"].astype(str))+"_"+(df["POS2"].astype(str))
df

,CHR,POS1,POS2,FREQUENCY,UNIFORMITY,TYPE,region
0,chr1,5262241,5290853,0.000000,0.000000,Low frequency unstable circle(LU),chr1:5262241_5290853
1,chr1,6682911,6714302,0.000000,0.000000,Low frequency unstable circle(LU),chr1:6682911_6714302
2,chr1,8074266,8101772,0.000000,0.000000,Low frequency unstable circle(LU),chr1:8074266_8101772
3,chr1,12191185,12227871,0.000000,0.000000,Low frequency unstable circle(LU),chr1:12191185_12227871
4,chr1,15536234,15563379,0.000000,0.000000,Low frequency unstable circle(LU),chr1:15536234_15563379
...,...,...,...,...,...,...,...
13066,chr8,133659488,133874959,0.384935,0.327195,High frequency stable circle(HS),chr8:133659488_133874959
13067,chr8,130566097,130910958,0.337267,0.303541,High frequency stable circle(HS),chr8:130566097_130910958
13068,chr8,121845273,121996460,0.375357,0.356589,High frequency stable circle(HS),chr8:121845273_121996460
13069,chr8,128547070,128647504,0.372092,0.372092,High frequency stable circle(HS),chr8:128547070_128647504


In [8]:
ECDNA_LIST = df[df['TYPE'].isin(['High frequency unstable circle(HU)', 'High frequency stable circle(HS)'])]['region'].tolist()
ECDNA_LIST

['chr12:28638746_28809102',
 'chr8:124063255_124169055',
 'chr8:138149259_138535657',
 'chr10:62691573_62850185',
 'chr10:66567893_66650247',
 'chr12:27788048_28082412',
 'chr8:118600715_119136453',
 'chr8:126305964_126487605',
 'chr8:132851685_132892345',
 'chr1:172976200_173170779',
 'chr8:127717002_127816732',
 'chr8:133659488_133874959',
 'chr8:130566097_130910958',
 'chr8:121845273_121996460',
 'chr8:128547070_128647504']

In [9]:
#fname = "COLO320DM_rep1"
fname = "PC3"

In [10]:
#fragments_path = "E:/05.project/04.ecDNA/01.data/CRC/GSM4861353_COLO320DM_rep2_atac_fragments.tsv.gz"
fragments_path = "E:/05.project/04.ecDNA/01.data/BGI_CL/PC3.fragments.tsv.gz"
df_fragments = read_bgi_as_dataframe(path=fragments_path)

### Transform: 删除chrM
df_fragments = Transform(df_fragments=df_fragments,Delete_chrM_option=True)

### segmentation 片段分割:
df_fragments_cutoff_segmentation = fragments_segmentation(df_fragments=df_fragments)

### Normalize(计算覆盖度Coverage)：
#单线程
df_fragments_cutoff_normalize = df_fragments_cutoff_segmentation.groupby(df_fragments_cutoff_segmentation["barcode"]).apply(Normalize)

### v3：：较于前版本v2，加入（合并）了:：cell_coverage.py 步骤 
df_fragments_cutoff_normalize_dd = df_fragments_cutoff_normalize.drop_duplicates(subset=['barcode','chr_100k'])
df_fragments_cutoff_normalize_dd.to_csv(f"./Fig-s1-a_ECDNA_CNV_BG/{fname}_cell_coverage.matrix.tsv",sep="\t",index=True)

df_fragments.chrom character content contains chr.


KeyboardInterrupt: 

In [ ]:
# 应用转换函数到每一行
dfcn_dd['chr_100k_hg38'] = dfcn_dd['chr_100k']
#dfcn_dd['chr_100k_hg38'] = dfcn_dd['chr_100k'].apply(convert_interval)
# 应用判断ECDNA函数，添加新列
dfcn_dd['is_ECDNA'] = dfcn_dd['chr_100k_hg38'].apply(has_overlap)

In [ ]:
dfcn_dd

In [ ]:
dfcn_dd['Group'] = dfcn_dd['is_ECDNA'].map({False: 'Background', True: 'ecDNA'})
df_tmp = dfcn_dd.drop_duplicates(subset=['chr_100k_hg38','Group'])
df_tmp["Group"].value_counts()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# 替换为你的真实数据
# dfcn_dd = pd.read_csv('your_data.tsv', sep='\t')

# 显式复制以避免修改原数据
plot_df = dfcn_dd.copy()
plot_df['Group'] = plot_df['is_ECDNA'].map({False: 'Background', True: 'ecDNA'})

# 分组
group_true = plot_df[plot_df['Group'] == 'ecDNA']['Coverage']
group_false = plot_df[plot_df['Group'] == 'Background']['Coverage']

# t检验
t_stat, p_val = ttest_ind(group_true, group_false, equal_var=False)

# 打印 p 值
print(f"t-test p-value: {p_val:.3e}")

# 绘图
plt.figure(figsize=(4, 5))
sns.boxplot(data=plot_df, x='Group', y='Coverage', palette="Set2")

# 添加 p 值标注
x1, x2 = 0, 1
y = plot_df['Coverage'].max() * 1.05
h = plot_df['Coverage'].max() * 0.05
plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c='k')

# p 值文本内容
if p_val == 0.0:
    p_text = "t-test, p < 1e-32"
else:
    p_text = f"t-test, p = {p_val:.2e}"

plt.text((x1 + x2) * 0.5, y + h * 1.1, p_text, ha='center', va='bottom', fontsize=12)

# 标签与标题
plt.xlabel("")  # 如果不需要横轴标题
plt.ylabel("Coverage")
plt.title(f"{fname}")
plt.tight_layout()
#plt.show()
plt.savefig(f"D:/02.project/18.ecDNA/02.code/v0.1.1/Fig-s1-a_ECDNA_CNV_BG/f01-t-test_{fname}.pdf", bbox_inches='tight')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 假设你已有 dfcn_dd，其中 'is_ECDNA' 是布尔型，'Coverage' 是数值型
# 设置图形风格
sns.set(style="whitegrid")
plt.figure(figsize=(8, 5))

# 绘制 TRUE 组密度曲线
sns.kdeplot(
    data=dfcn_dd[dfcn_dd['is_ECDNA'] == True],
    x="Coverage",
    fill=True,
    label='ecDNA',
    color='salmon',
    alpha=0.6
)

# 绘制 FALSE 组密度曲线
sns.kdeplot(
    data=dfcn_dd[dfcn_dd['is_ECDNA'] == False],
    x="Coverage",
    fill=True,
    label='Background',
    color='skyblue',
    alpha=0.6
)

plt.axvline(6, color = "b", linestyle = "-.")
plt.xlim(-3, 100)

# 添加图例和标题
plt.legend(title="Group")
plt.title(f"{fname}")
plt.xlabel("Coverage")
plt.ylabel("Density")
plt.tight_layout()
#plt.show()
plt.savefig(f"D:/02.project/18.ecDNA/02.code/v0.1.1/Fig-s1-a_ECDNA_CNV_BG/f02-density_{fname}.pdf", bbox_inches='tight')

In [ ]:
fragments_path = "E:/05.project/04.ecDNA/01.data/CRC/GSM4861353_COLO320DM_rep2_atac_fragments.tsv.gz"
df_fragments = read_bgi_as_dataframe(path=fragments_path)

### Transform: 删除chrM
df_fragments = Transform(df_fragments=df_fragments,Delete_chrM_option=True)

### segmentation 片段分割:
df_fragments_cutoff_segmentation = fragments_segmentation(df_fragments=df_fragments)

### Normalize(计算覆盖度Coverage)：
#单线程
df_fragments_cutoff_normalize = df_fragments_cutoff_segmentation.groupby(df_fragments_cutoff_segmentation["barcode"]).apply(Normalize)

### v3：：较于前版本v2，加入（合并）了:：cell_coverage.py 步骤 
df_fragments_cutoff_normalize_dd = df_fragments_cutoff_normalize.drop_duplicates(subset=['barcode','chr_100k'])
df_fragments_cutoff_normalize_dd.to_csv("./Fig-s1-a_ECDNA_CNV_BG/cell_coverage.matrix.tsv",sep="\t",index=True)